In [33]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [34]:
from dotenv import load_dotenv
import os 
from tavily import TavilyClient
from langchain.chat_models import init_chat_model
from sqlalchemy import create_engine, text
from langchain.tools import tool
load_dotenv()

True

In [35]:
GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [36]:
# Fast + Free
cheap_model = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

# Research + General Tasks
middle_model = init_chat_model(
    "google_genai:gemini-2.5-flash"
)

# Advanced Reasoning
advance_model = init_chat_model(
    "groq:llama-3.3-70b-versatile"
).with_fallbacks([middle_model, cheap_model])

In [37]:
from sqlalchemy import create_engine, text
from langchain_core.tools import tool

engine = create_engine(DATABASE_URL)

In [38]:
# user this information for testing porpose 
user_id_testing = "f5f7dea2-d2f9-431c-8529-aea5cd0fa49a"
user_mail = "nofackai@gmail.com"

In [39]:
import json
from typing import TypedDict, Annotated, List, Dict, Any, Sequence
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, AIMessage, ToolMessage
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from sqlalchemy import text

# ==========================================
# 1. Real Database & Context Tools
# ==========================================

@tool
def fetch_user_context(user_id: str) -> str:
    """Fetches the user's basic profile and memory of past conversations from the database using user_id."""
    try:
        query_profile = text("""SELECT full_name, email, role FROM users WHERE id = CAST(:user_id AS UUID);""")
        query_memory = text("""SELECT role, content FROM ai_memory WHERE user_id = CAST(:user_id AS UUID) ORDER BY created_at DESC LIMIT 5;""")
        
        with engine.connect() as conn:
            prof_res = conn.execute(query_profile, {"user_id": user_id}).fetchone()
            mem_res = conn.execute(query_memory, {"user_id": user_id}).fetchall()
            
            profile = dict(prof_res._mapping) if prof_res else {}
            memory = [dict(m._mapping) for m in mem_res] if mem_res else []
            
        return json.dumps({"profile": profile, "memory": memory}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})

@tool
def search_existing_leads(user_id: str) -> str:
    """Queries the database for existing leads belonging to the user. Returns file names and column schemas."""
    try:
        query = text("""
            SELECT id, file_name, status, row_count, columns
            FROM leads
            WHERE user_id = CAST(:user_id AS UUID)
            ORDER BY created_at DESC
            LIMIT 5;
        """)
        with engine.connect() as conn:
            result = conn.execute(query, {"user_id": user_id})
            rows = [dict(row._mapping) for row in result]
            
        if not rows:
            return "No matching leads found in database."
        
        for r in rows:
            r['id'] = str(r['id'])
        return json.dumps(rows, default=str)
    except Exception as e:
        return f"Error querying leads: {e}"

@tool
def search_existing_templates(user_id: str) -> str:
    """Queries the database for existing templates belonging to the user. Returns template names and variables."""
    try:
        query = text("""
            SELECT id, name, description, subject_line, variables, is_ai_generated
            FROM templates
            WHERE user_id = CAST(:user_id AS UUID)
            ORDER BY created_at DESC
            LIMIT 5;
        """)
        with engine.connect() as conn:
            result = conn.execute(query, {"user_id": user_id})
            rows = [dict(row._mapping) for row in result]
            
        if not rows:
            return "No matching templates found in database."
        
        for r in rows:
            r['id'] = str(r['id'])
        return json.dumps(rows, default=str)
    except Exception as e:
        return f"Error querying templates: {e}"

@tool
def call_lead_agent(target_audience: str) -> str:
    """Triggers the sub-Lead Agent to generate new leads when existing ones are insufficient."""
    return f"Lead Agent generation requested for {target_audience}."

@tool
def call_template_agent(role: str, skills: str) -> str:
    """Triggers the sub-Template Agent to create a new outreach template."""
    return f"Template Agent generation requested for {role}."

@tool
def propose_variable_mapping(lead_columns: list, template_variables: list) -> str:
    """Maps lead columns to template variables to prepare for campaign launch."""
    mapping = {var: f"Mapped to lead column '{var.strip('{}')}'" for var in template_variables}
    return json.dumps(mapping)

@tool
def send_test_email(user_id: str, template_id: str) -> str:
    """Simulates sending a test email for campaign verification."""
    return f"Test email for template successfully generated and queued for user review."

tools = [
    fetch_user_context,
    search_existing_leads,
    search_existing_templates,
    call_lead_agent,
    call_template_agent,
    propose_variable_mapping,
    send_test_email
]

# ==========================================
# 2. LangGraph State & Nodes
# ==========================================

class CampaignAgentState(MessagesState):
    user_id: str
    context_fetched: bool

def context_initializer(state: CampaignAgentState):
    """Ensures user context is loaded into the prompt conversation before the LLM acts."""
    if state.get("context_fetched"):
        return state
        
    context_data = fetch_user_context.invoke({"user_id": state["user_id"]})
    
    system_prompt = f"""
    You are the intelligent Campaign Orchestrator Agent. Your goal is to orchestrate the complete campaign creation process conversationally.
    
    CRITICAL RULE - STRICT STEP-BY-STEP EXECUTION:
    You MUST execute the campaign creation process strictly one step at a time. Do NOT combine steps. Do NOT jump ahead.
    You MUST ask the user for confirmation after EVERY single step and WAIT for their 'Yes' or 'No'.
    
    Step 1. Goal Confirmation: Check the memory below. Say "I see you want to target [goal]. Should we proceed with this?" (Wait for Yes)
    Step 2. Lead Selection: Call your lead tool. Show matching leads. Say "Use these or create new ones?" (Wait for Yes)
    Step 3. Template Selection: Call your template tool. Show matching templates. Say "Use this or create a new one?" (Wait for Yes)
    Step 4. Variable Mapping: Propose the mapping. Say "Here is the mapping. Confirm?" (Wait for Yes)
    Step 5. Campaign Testing: Call the 'send_test_email' tool. Say "I have sent a test email. Have you verified it looks good?" (Wait for Yes)
    Step 6. Launch: Say "Campaign launched successfully."
    
    AVAILABLE USER CONTEXT (Memory & Profile):
    {context_data}
    """
    
    return {
        "messages": [SystemMessage(content=system_prompt)],
        "context_fetched": True
    }

def campaign_orchestrator(state: CampaignAgentState):
    """The main brain of the campaign agent, utilizing the LiteLLM fallback gateway."""
    model_with_tools = advance_model.bind_tools(tools)
    try:
        response = model_with_tools.invoke(state["messages"])
        return {"messages": [response]}
    except Exception as e:
        return {"messages": [AIMessage(content="I'm sorry, I encountered a system error processing that request.")]}

tool_node = ToolNode(tools)

def route_tools(state: CampaignAgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return "tools"
    return END

# ==========================================
# 3. Compile Graph
# ==========================================

from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(CampaignAgentState)

workflow.add_node("init", context_initializer)
workflow.add_node("agent", campaign_orchestrator)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "init")
workflow.add_edge("init", "agent")
workflow.add_conditional_edges("agent", route_tools, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

memory = MemorySaver()
campaign_app = workflow.compile(checkpointer=memory)

# ==========================================
# 4. Interactive Testing Loop (Clean UI)
# ==========================================
def run_chatbot():
    config = {"configurable": {"thread_id": "strict_conversation_2"}}
    initial_state = {
        "messages": [],
        "user_id": user_id_testing,
        "context_fetched": False
    }

    print("========================================")
    print("Campaign Agent Chatbot (Clean Interface)")
    print("Type 'exit' or 'quit' to stop.")
    print("========================================\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'quit']:
            print("Ending conversation.")
            break
            
        input_dict = {"messages": [HumanMessage(content=user_input)]}
        if not initial_state["context_fetched"]:
            input_dict["user_id"] = initial_state["user_id"]
            input_dict["context_fetched"] = False
            initial_state["context_fetched"] = True
            
        for event in campaign_app.stream(input_dict, config=config):
            for node, values in event.items():
                if node == "agent":
                    last_message = values["messages"][-1]
                    if last_message.content:
                        print(f"\nCampaign Agent: {last_message.content}\n")

In [40]:
run_chatbot()

Campaign Agent Chatbot (Clean Interface)
Type 'exit' or 'quit' to stop.


Campaign Agent: I see you want to target AI and software companies for your Gen AI role. Should we proceed with this?


Campaign Agent: I see you want to target AI and software companies for a Gen AI internship. Should we proceed with this?

Next, I'll need to fetch your user context to get your basic profile and memory of past conversations. 



Campaign Agent: I'm sorry, I encountered a system error processing that request.


Campaign Agent: I'm sorry, I encountered a system error processing that request.

Ending conversation.
